# 🐍 Day 37 · Python Basics — HTTP Requests, Headers & JSON APIs (Deploying to the Cloud)

> **Goal of this notebook:** After deployment your agent lives at a URL. You talk to it with **HTTP requests**: a method, a URL, headers (with your key) and a JSON body.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. The parts of an HTTP request
2. Building URLs safely
3. Headers and API keys
4. Preparing a real request with `requests` (without sending it)
5. Status codes and error handling
6. 🔗 A client function for a deployed agent

## 1. The parts of an HTTP request
```
POST https://my-agent-abc.run.app/run           <- method + URL
Authorization: Bearer my-secret-key             <- headers
Content-Type: application/json
{"user_id": "u1", "message": "Hello agent"}      <- body (JSON)
```
- `GET` = read something, `POST` = send something to process.

## 2. Building URLs safely

In [ ]:
from urllib.parse import urljoin, urlencode, urlparse

base = "https://my-agent-abc.a.run.app/"
print(urljoin(base, "apps/podcast_agent/sessions"))

query = urlencode({"user_id": "u 1", "limit": 5})     # spaces and symbols are encoded for you
print(f"{base}history?{query}")

parts = urlparse("https://my-agent-abc.a.run.app/run?debug=true")
print(parts.scheme, parts.netloc, parts.path, parts.query)

## 3. Headers and API keys
The key comes from `.env`, never from the code. `api_server_cloud_run.py` checks it in the `X-API-Key` header.

In [ ]:
import os
os.environ.setdefault("API_SECRET_KEY", "demo-key-123")       # normally set in .env

headers = {
    "Content-Type": "application/json",
    "X-API-Key": os.getenv("API_SECRET_KEY"),
}
print({k: (v[:4] + "***" if "Key" in k else v) for k, v in headers.items()})   # hide the secret when printing

## 4. Prepare a real request (without sending it)
`requests` builds the exact bytes that would travel over the internet. `.prepare()` lets us look at them first.

In [ ]:
import requests, json

req = requests.Request(
    method="POST",
    url=urljoin(base, "run"),
    headers=headers,
    json={"user_id": "u1", "session_id": "s1", "message": "Make a 2-minute podcast about RAG"},
).prepare()

print(req.method, req.url)
print(req.headers["Content-Type"])
print(json.loads(req.body))
# To really send it you would write:  requests.post(url, headers=headers, json=body, timeout=30)

## 5. Status codes
| Code | Meaning |
|---|---|
| 200 | OK ✅ |
| 401 / 403 | Wrong or missing key 🔑 |
| 404 | Wrong URL |
| 429 | Too many requests — slow down |
| 500 | The server crashed |

In [ ]:
def explain(status):
    if 200 <= status < 300:
        return "success"
    messages = {401: "check your API key", 403: "not allowed", 404: "check the URL",
                429: "rate limited, wait and retry", 500: "server error, check the logs"}
    return messages.get(status, "unexpected status")

for code in [200, 401, 404, 429, 500, 418]:
    print(code, "->", explain(code))

## 6. 🔗 A client for a deployed agent
We pass in a **pretend `send` function** so the notebook works offline. In `test_deployed_agent_cloud_run.py` this is a real HTTP call.

In [ ]:
import time

def fake_send(request):
    fake_send.calls += 1
    if fake_send.calls == 1:
        return 429, {"error": "slow down"}          # first try: rate limited
    return 200, {"reply": "Here is your podcast script about RAG..."}
fake_send.calls = 0

def ask_agent(message, send=fake_send, retries=3):
    body = {"user_id": "u1", "message": message}
    for attempt in range(1, retries + 1):
        status, data = send(body)
        print(f"attempt {attempt}: {status} ({explain(status)})")
        if status == 200:
            return data["reply"]
        if status == 429:
            time.sleep(0.5 * attempt)              # wait a little longer each time (backoff)
            continue
        raise RuntimeError(f"Agent call failed: {status} {data}")
    raise RuntimeError("Gave up after retries")

print(ask_agent("Make a podcast about RAG"))

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Use `urlencode` to build the query string for `{"q": "vertex ai", "page": 2}`.
2. Prepare a `GET` request to `https://example.com/health` and print its method and URL.
3. What does `explain(403)` return?

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
print(urlencode({"q": "vertex ai", "page": 2}))
r = requests.Request("GET", "https://example.com/health").prepare()
print(r.method, r.url)
print(explain(403))   # not allowed
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Join URL | `urljoin(base, 'run')` |
| Query string | `urlencode({'a': 1})` |
| Headers | `{'X-API-Key': os.getenv('API_SECRET_KEY')}` |
| Build request | `requests.Request('POST', url, json=body).prepare()` |
| Send for real | `requests.post(url, json=body, timeout=30)` |
| Retry | `for attempt in range(1, 4): ... continue` |

## ➡️ What's next?
You now know the Python used in **Day_37_Deploy_ADK_to_Google_Cloud**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀